<a href="https://colab.research.google.com/github/JohnThefive/tinyml-data-preprocessing/blob/main/Tratamento_de_dados_tinyML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python tqdm

In [ ]:
import cv2
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
import kagglehub # A nova adição

# Configurações de diretório e resolução
OUTPUT_DIR = "./dataset_tinyml_96x96"
TARGET_SIZE = (96, 96)

def process_image(file_path):
    """
    Lê, redimensiona, converte para tons de cinza e salva na pasta correta.
    """
    filename = file_path.name

    try:
        parts = filename.split('_')
        if len(parts) < 5:
            return False

        eye_state = parts[4]

        if eye_state == '0':
            class_folder = "fechados"
        elif eye_state == '1':
            class_folder = "abertos"
        else:
            return False

        out_folder = Path(OUTPUT_DIR) / class_folder
        out_folder.mkdir(parents=True, exist_ok=True)
        out_path = out_folder / filename

        if out_path.exists():
            return True

        img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return False

        resized_img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(out_path), resized_img)
        return True

    except Exception as e:
        return False

def main():
    print("Fazendo o download ou localizando o dataset no cache do Kaggle...")
    # O kagglehub resolve o caminho automaticamente!
    INPUT_DIR = kagglehub.dataset_download("akashshingha850/mrl-eye-dataset")
    print(f"Dataset base localizado em: {INPUT_DIR}")

    print("Mapeando arquivos...")
    image_files = list(Path(INPUT_DIR).rglob("*.png"))
    total_files = len(image_files)

    print(f"Encontradas {total_files} imagens. Iniciando processamento paralelo...")

    with ProcessPoolExecutor() as executor:
        futures = {executor.submit(process_image, filepath): filepath for filepath in image_files}

        for future in tqdm(as_completed(futures), total=total_files, desc="Processando imagens"):
            pass

    print(f"\nConcluído! Dataset estruturado e pronto para uso em: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

Fazendo o download ou localizando o dataset no cache do Kaggle...
Using Colab cache for faster access to the 'mrl-eye-dataset' dataset.
Dataset base localizado em: /kaggle/input/mrl-eye-dataset
Mapeando arquivos...
Encontradas 84898 imagens. Iniciando processamento paralelo...


Processando imagens: 100%|██████████| 84898/84898 [00:20<00:00, 4095.56it/s]



Concluído! Dataset estruturado e pronto para uso em: ./dataset_tinyml_96x96


In [ ]:
from google.colab import files

files.download('/content/dataset_tinyml_96x96')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>